# 02 · Um lakehouse de vendas
**Pergunta:** quanto a loja vendeu por dia e quais registros não podemos usar?

Este projeto cria snapshots pequenos em tabelas Delta. Bronze é a fonte intacta; Silver tem a versão vigente e validada; Gold resume a regra de negócio. A quarentena conserva os motivos de rejeição. Isso é um exercício batch, não uma implementação de streaming.


In [ ]:
%run ./00_configuracao


In [ ]:
bronze = fonte()
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela("bronze_vendas"))
display(spark.table(tabela("bronze_vendas")).limit(10))


## Transformar sem perder a explicação
Chamamos `normalizar`, definida no 00. Preços têm no máximo duas casas decimais; quantidades são inteiros de 1 a 1.000. Cancelamento é um status válido. Uma chave só pode ter uma versão na Silver.


In [ ]:
silver, rejeitadas, substituidas = normalizar(spark.table(tabela("bronze_vendas")))
for nome, frame in [("silver_vendas", silver), ("quarentena_vendas", rejeitadas), ("versoes_substituidas", substituidas)]:
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela(nome))
display(rejeitadas)


## Dar uma forma útil para a análise
Gold considera somente pedidos concluídos. O valor do pedido é `quantidade × preço`. Receita não é lucro: não incluímos custo, frete, impostos ou margem neste dataset.


In [ ]:
gold = (silver.where("status = 'concluida'").groupBy("data_venda")
        .agg(F.count("venda_id").alias("pedidos"), F.sum("valor_centavos").alias("receita_centavos"))
        .withColumn("receita", F.col("receita_centavos") / 100)
        .withColumn("ticket_medio", F.col("receita") / F.col("pedidos")))
gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela("gold_diario"))
display(gold.orderBy("data_venda"))


In [ ]:
counts = {"bronze": bronze.count(), "silver": silver.count(), "rejeitadas": rejeitadas.count(), "substituidas": substituidas.count()}
assert counts == {"bronze": 727, "silver": 720, "rejeitadas": 6, "substituidas": 1}, counts
assert counts["bronze"] == sum(counts[k] for k in ["silver", "rejeitadas", "substituidas"])
assert silver.groupBy("venda_id").count().where("count > 1").count() == 0
assert gold.agg(F.sum("receita_centavos")).first()[0] == silver.where("status = 'concluida'").agg(F.sum("valor_centavos")).first()[0]
print(counts)


## Criar um painel na sua conta
No editor SQL, escolha o catálogo exibido pelo notebook e seu schema `lb_...`. Execute `SELECT * FROM gold_diario ORDER BY data_venda` e crie uma visualização de receita por dia. Salve a consulta em um dashboard disponível no seu workspace.

O painel local do repositório ajuda a experimentar as mesmas métricas. Ele não é uma captura de tela do Databricks.

**Desafio:** inclua uma regra para canais autorizados. Adicione uma linha com um canal desconhecido e explique por que ela foi rejeitada.
